# Real-fluid meanline design of a supersonic axial ORC turbine stage

**KCORC Summer School 2026 — student tutorial draft**  
**Format:** short explanations alternating with hands-on Python tasks  
**Main objective:** understand and implement the essential physics of a supersonic ORC turbine design point.  
**Final extension:** generate a small fixed-geometry off-design map for the following TESPy workshop.

By the end of the tutorial you should be able to connect

$$
(p_{0,\mathrm{in}},\, T_{0,\mathrm{in}},\, p_{\mathrm{out}},\, \dot{m})
\rightarrow \Delta h_{\mathrm{is}}
\rightarrow \text{sonic throat}
\rightarrow \text{velocity triangles}
\rightarrow P,\,\eta
\rightarrow \text{basic geometry}
\rightarrow \text{off-design map}
$$

## Workshop rhythm

| Block | Instructor input | Student work | Approx. time |
|---|---|---|---:|
| A | Real-fluid stage loading | Overall expansion and blade-speed ratio | 25 min |
| B | Choking and supersonic stator | Mach path, sonic throat, flow areas | 40 min |
| C | Impulse-stage velocity triangles | Stator/rotor triangles and Euler work | 45 min |
| D | Losses and geometry | Efficiency, partial admission, dimensions | 35 min |
| E | TESPy handoff | Small fixed-geometry map and CSV export | 20 min |

Orange cells marked **YOUR TASK** contain the equations you are expected to complete. Property calls, plotting, empirical correlations, and error handling are provided by the backend.

## 0. Environment and imports

From the repository root, the common Summer School setup is:

```bash
uv sync
```

In VS Code, select the Python interpreter from `.venv`. Run the next cell. If CoolProp or widgets are missing, the selected kernel is probably not the `uv` environment.

In [1]:
from __future__ import annotations

from dataclasses import replace
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from kcorc_turbine import (
    CoolPropBackend,
    DesignInputs,
    design_stage,
    evaluate_rotor_offdesign,
    evaluate_stator_offdesign,
    freeze_geometry,
    inlet_temperature_for_pressure,
)
from kcorc_turbine.checks import checkpoint, compare_with_reference
from kcorc_turbine.correlations import (
    disk_friction_loss_W,
    partial_admission_losses_W,
    rotor_velocity_coefficient,
    sector_filling_coefficient,
    stator_velocity_coefficient,
)
from kcorc_turbine.plotting import (
    plot_loss_breakdown,
    plot_nozzle_expansion,
    plot_offdesign_map,
    plot_velocity_triangles,
)
from kcorc_turbine.widgets import design_explorer, offdesign_explorer

backend = CoolPropBackend()
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True})

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root containing pyproject.toml")

REPO_ROOT = find_repo_root()
OUTPUT_DIR = REPO_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Repository root: {REPO_ROOT}")


Repository root: C:\GitHub\KCORC_summer_school


## 1. Start with the complete machine — interactive preview

Before deriving the model, explore the full reference implementation for a few minutes. Try changing only one input at a time.

Focus on three questions:

1. What makes the nozzle supersonic?
2. Why does changing rotational speed alter efficiency but not the prescribed cycle pressure ratio directly?
3. Why does lowering partial admission increase blade height?

This widget is a **demonstrator**, not a replacement for the calculations below.

In [2]:
default_inputs = DesignInputs()
design_explorer(default_inputs)

---
# Block A — From cycle boundary conditions to stage loading

The cycle supplies total inlet pressure and temperature, outlet pressure, and mass flow. The first ideal reference is an isentropic expansion:

$$
s_{\mathrm{out,is}} = s_{\mathrm{in}},
\qquad
\Delta h_{\mathrm{is}}
= h_{\mathrm{in}} - h_{\mathrm{out,is}},
\qquad
c_{\mathrm{is}}
= \sqrt{2\,\Delta h_{\mathrm{is}}}.
$$

The rotor blade speed at the mean diameter is

$$
U = \frac{\pi D_{\mathrm{mid}} n}{60}.
$$

and the stage-loading indicator used throughout the tutorial is 
$$
\frac{U}{c_{\mathrm{is}}}.
$$

In [3]:
inputs = DesignInputs(
    fluid="R1233zd(E)",
    p_in_Pa=2000e3,
    T_in_K=150.0 + 273.15,
    p_out_Pa=500e3,
    m_dot_kg_s=5.025,
    n_rpm=11000.0,
    D_mid_m=0.2,
    alpha_stator_deg=11.5,
    beta_rotor_in_deg=23.0,
    beta_rotor_out_deg=25.0,
    partial_admission=1.0,
    rotor_solidity=1.0,
    rotor_chord_m=0.025,
)
inputs

DesignInputs(fluid='R1233zd(E)', p_in_Pa=2000000.0, T_in_K=423.15, p_out_Pa=500000.0, m_dot_kg_s=5.025, n_rpm=11000.0, D_mid_m=0.2, alpha_stator_deg=11.5, beta_rotor_in_deg=23.0, beta_rotor_out_deg=25.0, partial_admission=1.0, rotor_solidity=1.0, rotor_chord_m=0.025, stator_solidity=1.2, rotor_height_clearance_m=0.0003)

<div style="padding:0.8rem 1rem;background:#fff3e0;border-left:6px solid #ef6c00;border-radius:5px">
<b>YOUR TASK A1 — overall expansion</b><br>
Use the prepared property backend and implement the five equations below. Keep all calculations in SI units.
</div>

In [4]:
# Thermodynamic states supplied by the property backend
state_in = backend.state_PT(inputs.p_in_Pa, inputs.T_in_K, inputs.fluid)
state_out_is = backend.state_PS(inputs.p_out_Pa, state_in["s_J_kgK"], inputs.fluid)

# TODO A1: replace every `...` with the appropriate expression.
pressure_ratio = ...
dh_is_J_kg = ...
c_is_m_s = ...
U_m_s = ...
U_over_cis = ...

{
    "pressure_ratio": pressure_ratio,
    "dh_is_kJ_kg": dh_is_J_kg / 1000,
    "c_is_m_s": c_is_m_s,
    "U_m_s": U_m_s,
    "U_over_cis": U_over_cis,
}

TypeError: unsupported operand type(s) for /: 'ellipsis' and 'int'

In [ ]:
checkpoint("Pressure ratio", pressure_ratio, (3, 6))
checkpoint("Isentropic velocity", c_is_m_s, (170, 300), "m/s")
checkpoint("Blade-speed ratio U/c_is", U_over_cis, (0.20, 0.60))

True

### Interpretation checkpoint

- At fixed diameter and cycle conditions, what happens to \(U/c_{is}\) when speed doubles?
- Does the pressure ratio change just because the shaft speed changes?
- Why is $\displaystyle \frac{U}{c_{\mathrm{is}}}$ usually a more useful similarity parameter than rpm alone?

---
# Block B — Quasi-1D supersonic stator and sonic throat

Along an ideal adiabatic nozzle, the local static state is evaluated at constant entropy. The conversion of enthalpy into velocity gives

$$
c(p)=\sqrt{2\left(h_0-h(p,s_0)\right)},
\qquad Ma(p)=\frac{c(p)}{a(p,s_0)}.
$$

The sonic throat is the first position where $\displaystyle Ma\geq 1$. For the supplied pressure ratio, the nozzle is convergent-divergent.

In [ ]:
# The backend performs only the repetitive real-fluid state calls here.
p_path = np.linspace(inputs.p_in_Pa, inputs.p_out_Pa, 220)
path_records = []
for p_local in p_path:
    st = backend.state_PS(float(p_local), state_in["s_J_kgK"], inputs.fluid)
    path_records.append(st)

nozzle_path_student = pd.DataFrame(path_records)
nozzle_path_student["pressure_ratio_local"] = p_path / inputs.p_out_Pa
nozzle_path_student.head(3)

<div style="padding:0.8rem 1rem;background:#fff3e0;border-left:6px solid #ef6c00;border-radius:5px">
<b>YOUR TASK B1 — Mach-number path and throat</b><br>
Compute the local velocity and Mach number for every row, then identify the first sonic point.
</div>

In [ ]:
# TODO B1
nozzle_path_student["c_is_m_s"] = ...
nozzle_path_student["Ma_is"] = ...

sonic_indices = np.flatnonzero(...)
if len(sonic_indices) == 0:
    throat_index = int(...)
    is_choked = False
else:
    throat_index = int(...)
    is_choked = True

throat_student = nozzle_path_student.iloc[throat_index]
print(f"Choked: {is_choked}")
print(f"Throat pressure: {throat_student['p_Pa']/1e3:.1f} kPa")
print(f"Exit Mach (isentropic): {nozzle_path_student.iloc[-1]['Ma_is']:.3f}")

In [ ]:
checkpoint("Throat pressure", float(throat_student["p_Pa"] / 1e3), (1500, 1700), "kPa")
checkpoint("Ideal nozzle exit Mach", float(nozzle_path_student.iloc[-1]["Ma_is"]), (1.65, 1.90))
plot_nozzle_expansion(nozzle_path_student, throat_index)
plt.show()

## Stator losses and the real-fluid outlet state

So far, we have assumed an **isentropic** nozzle expansion. In a real turbine, viscous losses reduce the kinetic energy obtained from the available enthalpy drop.

The detailed origin of the empirical nozzle-loss correlation will be discussed during the lecture. For this tutorial, you do **not** need to implement or copy the polynomial correlation. The backend evaluates the stator velocity coefficient, denoted by $\varphi_s$.

The actual nozzle outlet enthalpy is then obtained from

$$
h_1
=
h_0
-
\varphi_s^2
\left(
h_0-h_{1,\mathrm{is}}
\right).
$$


In [ ]:
Ma_out_is = float(nozzle_path_student.iloc[-1]["Ma_is"])
phi_stator = stator_velocity_coefficient(Ma_out_is)
h_out_is = float(nozzle_path_student.iloc[-1]["h_J_kg"])

# This part is supplied because the property inversion itself is not the learning objective.
h_nozzle_out_J_kg = state_in["h_J_kg"] - phi_stator**2 * (state_in["h_J_kg"] - h_out_is)
state_nozzle_out = backend.state_PH(inputs.p_out_Pa, h_nozzle_out_J_kg, inputs.fluid)
c_nozzle_out_m_s = np.sqrt(2.0 * (state_in["h_J_kg"] - h_nozzle_out_J_kg))
Ma_nozzle_out = c_nozzle_out_m_s / state_nozzle_out["a_m_s"]

pd.Series({
    "phi_stator": phi_stator,
    "Ma_out_is": Ma_out_is,
    "Ma_out_loss_adjusted": Ma_nozzle_out,
    "rho_nozzle_out_kg_m3": state_nozzle_out["rho_kg_m3"],
})

NameError: name 'nozzle_path_student' is not defined

<div style="padding:0.8rem 1rem;background:#fff3e0;border-left:6px solid #ef6c00;border-radius:5px">
<b>YOUR TASK B2 — throat and outlet areas</b><br>
Use continuity, \(A=\dot m/(
ho c)\), at the sonic throat and at the loss-adjusted outlet.
</div>

In [ ]:
# TODO B2
A_throat_m2 = ...
A_outlet_m2 = ...
area_ratio = ...

pd.Series({
    "A_throat_mm2": A_throat_m2 * 1e6,
    "A_outlet_mm2": A_outlet_m2 * 1e6,
    "A_outlet_over_A_throat": area_ratio,
})

In [ ]:
checkpoint("Total throat area", A_throat_m2 * 1e6, (420, 510), "mm²")
checkpoint("Loss-adjusted total outlet area", A_outlet_m2 * 1e6, (850, 1050), "mm²")

## Partial admission and first-order blade height

The stator exit velocity is resolved into axial and tangential components:

\[
c_{1a}=c_1\sin\alpha_1,\qquad c_{1u}=c_1\cos\alpha_1.
\]

The active annulus flow area is approximately \(e\pi D_{mid}h\), giving

\[
h_s=\frac{\dot m}{\rho_1c_{1a}e\pi D_{mid}}.
\]

The integer number of nozzles changes the effective admission fraction slightly. The pitch calculation is prepared; you will complete the velocity components and height.

In [ ]:
alpha_1_rad = np.radians(inputs.alpha_stator_deg)
chord_stator_ax_m = 1.25 * inputs.rotor_chord_m
stator_pitch_m = chord_stator_ax_m / inputs.stator_solidity
circumference_m = np.pi * inputs.D_mid_m
no_nozzles = max(1, round(inputs.partial_admission * circumference_m / stator_pitch_m))
partial_admission_eff = no_nozzles * stator_pitch_m / circumference_m

# TODO B3
c1a_m_s = ...
c1u_m_s = ...
h_nozzle_m = ...
h_rotor_m = ...

pd.Series({
    "c1a_m_s": c1a_m_s,
    "c1u_m_s": c1u_m_s,
    "no_nozzles": no_nozzles,
    "partial_admission_eff": partial_admission_eff,
    "h_nozzle_mm": h_nozzle_m * 1e3,
    "h_rotor_mm": h_rotor_m * 1e3,
})

In [ ]:
checkpoint("Nozzle height", h_nozzle_m * 1e3, (8.5, 12.5), "mm")

> **STOP — joint discussion.** Why is the blade height based on the axial component \(c_{1a}\), while the throat/outlet slot areas above were defined perpendicular to the local nozzle velocity?

---
# Block C — Rotor velocity triangles and Euler work

At rotor inlet:

\[
w_{1a}=c_{1a},\qquad w_{1u}=c_{1u}-U.
\]

The rotor relative outlet speed is reduced by the prepared empirical coefficient \(\varphi_r\) and a partial-admission filling/emptying coefficient \(K_s\). The rotor metal exit angle is represented internally as \(\beta_2=180^\circ-\beta_{2,input}\).

<div style="padding:0.8rem 1rem;background:#fff3e0;border-left:6px solid #ef6c00;border-radius:5px">
<b>YOUR TASK C1 — rotor inlet triangle</b><br>
Transform the stator exit velocity into the relative frame and calculate the relative flow angle.
</div>

In [ ]:
# TODO C1
w1a_m_s = ...
w1u_m_s = ...
w1_m_s = ...
beta1_flow_deg = ...
Ma1_rel = ...

pd.Series({
    "w1_m_s": w1_m_s,
    "beta1_flow_deg": beta1_flow_deg,
    "beta1_metal_deg": inputs.beta_rotor_in_deg,
    "incidence_proxy_deg": beta1_flow_deg - inputs.beta_rotor_in_deg,
    "Ma1_rel": Ma1_rel,
})

The empirical rotor coefficient is supplied. You still implement the velocity transformation back to the absolute frame.

In [ ]:
beta2_geometry_deg = 180.0 - inputs.beta_rotor_out_deg
theta_deg = beta2_geometry_deg - inputs.beta_rotor_in_deg
phi_rotor_base, phi_rotor = rotor_velocity_coefficient(
    theta_deg, Ma1_rel, h_rotor_m, inputs.rotor_chord_m
)

rotor_pitch_nominal_m = inputs.rotor_chord_m / inputs.rotor_solidity
no_rotor_blades = max(3, round(circumference_m / rotor_pitch_nominal_m))
rotor_pitch_m = circumference_m / no_rotor_blades
active_arc_m = partial_admission_eff * circumference_m
Ks = sector_filling_coefficient(rotor_pitch_m, active_arc_m)

# TODO C2
w2_signed_m_s = ...
w2a_m_s = ...
w2u_m_s = ...
c2a_m_s = ...
c2u_m_s = ...
c2_m_s = ...

student_velocities = {
    "U_m_s": U_m_s,
    "c1u_m_s": c1u_m_s,
    "c1a_m_s": c1a_m_s,
    "w1u_m_s": w1u_m_s,
    "w1a_m_s": w1a_m_s,
    "w2u_m_s": w2u_m_s,
    "w2a_m_s": w2a_m_s,
    "c2u_m_s": c2u_m_s,
    "c2a_m_s": c2a_m_s,
}
plot_velocity_triangles(student_velocities)
plt.show()

In [ ]:
checkpoint("Relative inlet Mach", Ma1_rel, (0.85, 1.25))
checkpoint("Rotor exit tangential velocity c2u", c2u_m_s, (-45, 5), "m/s")

### Engineering interpretation

1. Why does non-zero \(c_{2u}\) represent kinetic energy not fully converted to shaft work?
2. Is a small negative \(c_{2u}\) necessarily unacceptable if a downstream machine is connected through a sufficiently long duct?
3. Why can changing only the rotor exit metal angle have a surprisingly small effect on \(c_{2u}\)?

---
# Block D — Power, losses, efficiency, and geometry

Euler's turbine equation gives the aerodynamic power:

\[
P_{aero}=\dot m U(c_{1u}-c_{2u}).
\]

The teaching model then subtracts disk-friction and partial-admission loss estimates. These empirical correlations are provided because their calibration and uncertainty are a separate research topic.

In [ ]:
# The rotor static outlet state is needed by the empirical power-loss terms.
dh_rotor_static_J_kg = 0.5 * (w1_m_s**2 - w2_signed_m_s**2)
h_rotor_out_J_kg = h_nozzle_out_J_kg + dh_rotor_static_J_kg
state_rotor_out = backend.state_PH(inputs.p_out_Pa, h_rotor_out_J_kg, inputs.fluid)

# TODO D1
P_aero_W = ...
P_fric_W = disk_friction_loss_W(...)
P_pa = partial_admission_losses_W(
    partial_admission=...,
    rho2_kg_m3=...,
    n_rpm=...,
    D_mid_m=...,
    h_rotor_m=...,
    U_m_s=...,
)
P_partial_admission_W = ...
P_mech_W = ...
eta_turb = ...

pd.Series({
    "P_aero_kW": P_aero_W / 1e3,
    "P_fric_kW": P_fric_W / 1e3,
    "P_partial_admission_kW": P_partial_admission_W / 1e3,
    "P_mech_kW": P_mech_W / 1e3,
    "eta_turb": eta_turb,
})

In [ ]:
checkpoint("Aerodynamic power", P_aero_W / 1e3, (85, 110), "kW")
checkpoint("Mechanical power", P_mech_W / 1e3, (75, 100), "kW")
checkpoint("Turbine efficiency", eta_turb, (0.45, 0.68))

<div style="padding:0.8rem 1rem;background:#fff3e0;border-left:6px solid #ef6c00;border-radius:5px">
<b>YOUR TASK D2 — geometric summary</b><br>
Complete the hub/tip diameters and the per-nozzle throat/outlet widths.
</div>

In [ ]:
# TODO D2
D_hub_m = ...
D_tip_m = ...
b_nozzle_throat_m = ...
b_nozzle_out_m = ...

student_summary = pd.DataFrame([
    ("Pressure ratio", pressure_ratio, "-"),
    ("Ideal nozzle exit Mach", Ma_out_is, "-"),
    ("Real nozzle exit Mach", Ma_nozzle_out, "-"),
    ("U/c_is", U_over_cis, "-"),
    ("Mechanical power", P_mech_W / 1e3, "kW"),
    ("Turbine efficiency", eta_turb, "-"),
    ("Nozzle count", no_nozzles, "-"),
    ("Rotor blade count", no_rotor_blades, "-"),
    ("Rotor height", h_rotor_m * 1e3, "mm"),
    ("Hub diameter", D_hub_m * 1e3, "mm"),
    ("Tip diameter", D_tip_m * 1e3, "mm"),
    ("Single-nozzle throat width", b_nozzle_throat_m * 1e3, "mm"),
    ("Single-nozzle outlet width", b_nozzle_out_m * 1e3, "mm"),
], columns=["Quantity", "Value", "Unit"])
student_summary

## Compare your reconstruction with the complete reference model

The reference model uses the same equations but also contains robust state handling and a consistent data structure for the interactive and off-design parts.

In [ ]:
reference_design = design_stage(inputs, backend=backend)
compare_with_reference("Efficiency", eta_turb, reference_design.performance["eta_turb"], unit="-")
compare_with_reference("Mechanical power", P_mech_W, reference_design.performance["P_mech_W"], unit="W")
compare_with_reference("Throat area", A_throat_m2, reference_design.geometry["A_throat_m2"], unit="m²")
reference_design.summary()

In [ ]:
design_file = reference_design.export_json(OUTPUT_DIR / "design_point.json")
print(f"Saved frozen design point to {design_file}")

In [ ]:
plot_loss_breakdown(reference_design)
plt.show()

## Design playground — use it after completing the equations

Change **one parameter only**, predict the direction of change before moving the slider, and then explain the result.

Suggested group cases:

- lower rotational speed;
- lower mean diameter;
- higher pressure ratio;
- higher partial admission;
- different stator angle;
- different working fluid.

In [ ]:
design_explorer(inputs)

## What the final reduced-order output can look like

The figure below is an example generated by the research code. During the tutorial we create a smaller map with the same basic interpretation.

![Example off-design efficiency map](../assets/offdesign_example.png)


---
# Block E — Small fixed-geometry off-design map for TESPy

The geometry is now frozen. This distinction is essential:

- **design calculation:** geometry may be sized from \(\dot m,p,T\);
- **off-design calculation:** throat area, outlet area, diameter, blade height, angles, and blade counts stay fixed.

For each pressure ratio, the fixed stator determines its flow capacity. Rotational speed then changes the rotor velocity triangle and efficiency.

This final section is deliberately short. Its purpose is to create a machine-readable handoff for the following TESPy workshop.

In [ ]:
geometry = freeze_geometry(reference_design)

# A small grid is sufficient during the live tutorial.
p_critical_Pa = backend.critical_pressure(geometry.fluid)
PR_upper = min(1.28 * geometry.pressure_ratio_design, 0.94 * p_critical_Pa / geometry.p_out_design_Pa)
PR_values = np.linspace(0.60 * geometry.pressure_ratio_design, PR_upper, 9)
n_values = np.linspace(0.60 * geometry.n_design_rpm, 2.20 * geometry.n_design_rpm, 13)

print(f"Map size: {len(PR_values)} x {len(n_values)} = {len(PR_values)*len(n_values)} points")

<div style="padding:0.8rem 1rem;background:#fff3e0;border-left:6px solid #ef6c00;border-radius:5px">
<b>YOUR TASK E1 — build the map efficiently</b><br>
The stator calculation depends on pressure ratio but not on rotational speed. Evaluate it once per pressure ratio, then evaluate the rotor for all speeds.
</div>

In [ ]:
records = []

# TODO E1: complete the two nested loops.
for PR in PR_values:
    p_in_map_Pa = ...
    T_in_map_K = ...
    stator_point = ...

    for n_map_rpm in n_values:
        point = ...
        records.append(...)

map_df = pd.DataFrame.from_records(records)
map_df.head()

In [ ]:
plot_offdesign_map(
    map_df,
    design_PR=geometry.pressure_ratio_design,
    design_n_rpm=geometry.n_design_rpm,
)
plt.show()

<div style="padding:0.8rem 1rem;background:#fff3e0;border-left:6px solid #ef6c00;border-radius:5px">
<b>YOUR TASK E2 — non-dimensional TESPy handoff</b><br>
Create normalized pressure ratio, speed, mass flow, and power columns.
</div>

In [ ]:
# TODO E2
map_df["PR_rel"] = ...
map_df["n_rel"] = ...
map_df["m_dot_rel"] = ...
map_df["P_mech_rel"] = ...

export_columns = [
    "pressure_ratio", "n_rpm", "PR_rel", "n_rel",
    "p_in_Pa", "T_in_K", "p_out_Pa",
    "m_dot_kg_s", "m_dot_rel",
    "eta_turb", "P_mech_W", "P_mech_rel",
    "Ma_nozzle_out", "choked", "area_mismatch_rel", "valid",
]

map_file = OUTPUT_DIR / "turbine_map_student.csv"
map_df[export_columns].to_csv(map_file, index=False)
print(f"Saved {map_file.resolve()}")

## Explore the interpolated map

This is the reduced-order interface the cycle model can query. Points outside the valid map region must not be extrapolated blindly.

In [ ]:
offdesign_explorer(
    map_df,
    design_PR=geometry.pressure_ratio_design,
    design_n_rpm=geometry.n_design_rpm,
)

## Final discussion and TESPy handoff

Be prepared to answer:

1. Which map inputs are prescribed by the cycle and which can be controlled by the turbine/generator?
2. Why is mass flow not completely independent for a fixed choked nozzle?
3. Why must off-design geometry remain frozen?
4. Which regions of the map would require CFD or experimental validation before engineering use?
5. What information does TESPy need from the map: efficiency only, or also mass-flow capacity and validity limits?

### Model limitations

- quasi-one-dimensional stator treatment;
- empirical stator, rotor, disk-friction, and partial-admission correlations;
- no explicit shock/boundary-layer interaction;
- no radial equilibrium or 3D effects;
- no leakage or detailed tip-clearance model;
- map intended for teaching and reduced-order cycle coupling, not final hardware certification.

### Suggested references

- J. Špale, PhD thesis (2024), methodology and 1D design of an axial supersonic ORC turbine stage.